# The Wave Equation

$u_{tt} = c^2 u_{xx}$ describes a string, a sound wave or a signal on a line. Unlike diffusion, it keeps shapes sharp and moves them at speed $c$. This notebook solves it by the method of lines, checks d'Alembert's picture of travelling waves, writes a leapfrog scheme to meet the **CFL condition**, and tracks energy.

In [ ]:
using OrdinaryDiffEq, SparseArrays, LinearAlgebra, Plots
c, n = 1.0, 199
h = 1 / (n + 1); xs = h .* (1:n)
A = spdiagm(-1 => ones(n - 1), 0 => fill(-2.0, n), 1 => ones(n - 1)) ./ h^2   # fixed ends: u = 0
wave(w, p, t) = [w[n+1:end]; c^2 .* (A * w[1:n])]   # state w = [u; u_t]
u0 = exp.(-200 .* (xs .- 0.5) .^ 2)
sol = solve(ODEProblem(wave, [u0; zeros(n)], (0.0, 2.0)), Vern9(); reltol=1e-10, abstol=1e-12)

## Travelling and reflecting

d'Alembert: every solution is $F(x - ct) + G(x + ct)$. A pulse released from rest splits into two half-height pulses moving apart. At a fixed end each reflects **upside down**, so at $t = 1$ they meet as $-u_0$. At $t = 2$ the original shape is back.

In [ ]:
plot(xs, [sol(t)[1:n] for t in (0.0, 0.2, 0.45, 0.7, 1.0)], lw=2, label=["t = 0" "t = 0.2" "t = 0.45" "t = 0.7" "t = 1"], xlabel="x", ylabel="u", title="A pulse splits, reflects and inverts")

In [ ]:
(error_t1_vs_minus_u0 = maximum(abs.(sol(1.0)[1:n] .+ u0)), error_t2_vs_u0 = maximum(abs.(sol(2.0)[1:n] .- u0)))

In [ ]:
ts = range(0, 2, length=300)
heatmap(xs, ts, permutedims(reduce(hcat, [sol(t)[1:n] for t in ts])), c=:balance, xlabel="x", ylabel="t", title="Characteristics: lines of slope 1/c")

## The CFL condition

The classic explicit scheme for waves is **leapfrog**:

$$u^{k+1} = 2u^k - u^{k-1} + (c\,\Delta t)^2 A u^k.$$

Courant, Friedrichs and Lewy (1928) showed it is stable only if the **Courant number** $c\,\Delta t/h \le 1$: numerical information must travel at least as fast as the wave.

In [ ]:
function leapfrog(courant, T)
    dt = courant * h / c
    prev = u0
    curr = u0 .+ (dt^2 / 2) .* c^2 .* (A * u0)   # first step from rest
    for _ in 2:round(Int, T / dt)
        prev, curr = curr, 2 .* curr .- prev .+ (c * dt)^2 .* (A * curr)
    end
    return curr
end
plot(plot(xs, leapfrog(0.9, 1.0), lw=2, label="", title="Courant number 0.9"),
     plot(xs, leapfrog(1.02, 1.0), lw=2, label="", title="Courant number 1.02"), layout=(1, 2), size=(900, 330), xlabel="x")

## Energy

The energy $E = \tfrac12\int (u_t^2 + c^2u_x^2)\,dx$ is constant for the exact equation. Its discrete version is $\tfrac h2(v\cdot v - c^2\,u\cdot Au)$. An accurate explicit solver keeps it constant; the implicit FBDF, built for damped problems, bleeds energy away.

In [ ]:
energy(w) = h / 2 * (dot(w[n+1:end], w[n+1:end]) - c^2 * dot(w[1:n], A * w[1:n]))
long = [name => solve(ODEProblem(wave, [u0; zeros(n)], (0.0, 20.0)), alg; reltol=1e-6) for (name, alg) in ("Tsit5" => Tsit5(), "FBDF" => FBDF())]
ts = range(0, 20, length=200)
plot(ts, [[energy(s(t)) / energy(s(0.0)) for t in ts] for (_, s) in long], lw=2, label=permutedims(first.(long)), xlabel="t", ylabel="energy / initial energy", title="Energy over ten round trips")

## Standing waves and harmonics

With fixed ends, $\sin(k\pi x)\cos(k\pi c t)$ is a solution: a standing wave that vibrates at frequency $kc/2$, the $k$-th harmonic. A plucked string is a sum of harmonics.

In [ ]:
standing = solve(ODEProblem(wave, [sin.(2pi .* xs); zeros(n)], (0.0, 1.0)), Vern9(); reltol=1e-10, abstol=1e-12)
plot(xs, [standing(t)[1:n] for t in range(0, 0.5, length=6)], lw=2, label="", xlabel="x", title="Second harmonic: nodes stay still")

## Exercises

1. Release the string with velocity $u_t = \exp(-200(x - 0.5)^2)$ instead of displacement. Do the pulses still meet inverted at $t = 1$?
2. With no-flux ends ($u_x = 0$), pulses reflect **without** inverting. Change $A$ and check.
3. Coarsen to $n = 49$ and repeat the $t = 2$ return test. The error comes from **dispersion**: short waves travel too slowly on the grid. Look for trailing ripples.
4. Find the largest stable Courant number for leapfrog to two decimals.